# 20장 · RAG의 근거 문서와 기준 질문 준비하기

가상 도서관의 휴관일을 묻는 사람에게 답하려면 어떤 문서를 확인해야 할까요? 모델에 질문하기 전, 독자가 다시 찾아볼 수 있는 원문과 쪽수를 준비합니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 pypdf 6.21.4, reportlab 4.4.9입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.

## 답이 있는 위치 먼저 찾기

RAG는 질문을 받았을 때 관련 자료를 찾아 그 내용을 답변 입력에 넣는 방식입니다. 원문을 모아 이름과 위치를 붙이는 일은 미리 해 두고, 어느 자료를 가져올지는 질문을 받은 뒤 결정합니다.


### 셀 01 · 둘째 쪽의 근거 읽기

이 수업의 공통 문서 세 건은 실제 도서관 정책이 아닌 가상 자료입니다.

5장의 JSON은 학습 기록 한 건이라 제목은 title에서 읽었습니다. 오늘은 아래에서 제공하는 도서관 문서 목록을 사용합니다. documents[0]["text"]에는 대출 기간, documents[1]["text"]에는 휴관 안내가 들어 있습니다.

In [1]:
documents = [
  {"id":"library-1","text":"도서 대출 기간은 14일입니다.","source":"가상 도서관 이용 안내","page":1},
  {"id":"library-2","text":"도서관은 월요일에 휴관합니다.","source":"가상 도서관 이용 안내","page":2},
  {"id":"library-3","text":"무인 반납함은 24시간 이용할 수 있습니다.","source":"가상 도서관 이용 안내","page":3}
]
closed = documents[1]
print(closed["id"], closed["page"], closed["text"])

library-2 2 도서관은 월요일에 휴관합니다.


library-2, 2쪽, 월요일 휴관이라는 내용이 한 건에 붙어 있습니다. 같은 자료를 다음 장에서 나눠도 이 출처를 되찾을 수 있어야 합니다.

### 과제 02 · 대출 기간 질문의 근거 고르기

‘대출 기간은 며칠인가요?’의 기대 원문 ID와 본문을 documents에서 찾아 출력하세요.

In [2]:
# 문서 한 건을 골라 id와 text를 읽으세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>loan = next(d for d in documents if &quot;대출 기간&quot; in d[&quot;text&quot;])
print(loan[&quot;id&quot;], loan[&quot;text&quot;])</code></pre>
<p>실행 결과</p><pre><code>library-1 도서 대출 기간은 14일입니다.</code></pre>
<p>library-1을 찾으면 14일이라는 수치까지 원문에서 확인할 수 있습니다. 단어를 찾는 이 짧은 코드는 이번 자료를 읽기 위한 연습입니다.</p>
</details>

## 텍스트와 PDF를 읽은 뒤 위치 확인하기

텍스트 파일 한 개와 PDF의 쪽 하나는 위치 정보가 다릅니다. PDF를 글자로 추출한 다음에는 실제 쪽의 문장과 대조해야 합니다. 글자가 읽히지 않은 스캔 페이지를 빈 내용 그대로 검색에 넣지 않습니다.


### 셀 03 · 텍스트 파일을 한 문서로 읽기

제공 셀이 실제 임시 UTF-8 파일에 도서관 안내를 넣습니다. 원문을 읽은 뒤 파일 이름을 출처로 보관합니다.

In [3]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as folder:
    path = Path(folder) / "library.txt"
    path.write_text(documents[0]["text"], encoding="utf-8")
    plain = {"text": path.read_text(encoding="utf-8").strip(), "source": path.name, "page": None}
print(plain["source"], plain["page"])

library.txt None


library.txt와 None이 나옵니다. 페이지 구분이 없는 텍스트에 임의로 ‘PDF 1쪽’이라고 붙이지 않습니다.

### 셀 04 · PDF 추출 결과에서 쪽 번호 확인하기

제공된 예제 PDF의 첫 쪽을 이미 읽어 보관한 결과입니다. 이번 셀은 추출 기록을 읽습니다.

In [4]:
extracted = {"id": "library-sample-1", "source": "library-sample.pdf", "page": 1,
             "text": "Fictional library: Books may be borrowed for 14 days."}
print(extracted["page"], extracted["text"])

1 Fictional library: Books may be borrowed for 14 days.


1쪽에 대출 기간 14일이 적혀 있습니다. 이 PDF의 영문 문장과 공통 한국어 문서는 별도 자료이므로 ID를 바꿔 쓰지 않습니다. 실제 예제의 load_pdf는 PdfReader의 pages를 순회해 extract_text 결과와 쪽 번호를 함께 저장합니다.

### 셀 05 · 메모리에서 한 쪽짜리 PDF를 열어 보기

수업 커널에 reportlab과 pypdf가 있으면 제공 코드가 가상 도서관의 대출 문장 한 줄을 실제 PDF로 만듭니다. PDF를 다시 열어 첫 번째 쪽에서 추출한 글이 원래 문장과 맞는지 보세요.

In [5]:
from io import BytesIO
from importlib.util import find_spec

if find_spec("reportlab") and find_spec("pypdf"):
    from reportlab.pdfgen import canvas
    from pypdf import PdfReader
    pdf = BytesIO()
    sheet = canvas.Canvas(pdf)
    sheet.drawString(72, 720, extracted["text"])
    sheet.showPage()
    sheet.save()
    pdf.seek(0)
    first_page = PdfReader(pdf).pages[0]
    print(first_page.extract_text().strip())
else:
    print("직접 PDF를 읽으려면 reportlab과 pypdf가 설치된 커널에서 실행하세요.")

Fictional library: Books may be borrowed for 14 days.


만든 PDF에서 텍스트를 추출하면 첫 쪽의 14 days가 그대로 나옵니다. pages[0]은 사람이 보는 1쪽입니다. 텍스트가 있는 학습용 PDF를 확인한 결과이며, 스캔 이미지만 있는 PDF까지 읽었다는 뜻은 아닙니다.

## 자료에서 알 수 없는 질문 남겨 두기

기준 질문에는 답을 뒷받침할 문서 ID와 현재 묶음에서 답할 수 있는지 여부를 함께 적습니다. 주차 요금은 세 문서 어디에도 없습니다.


### 과제 06 · 주차 질문을 기준 질문으로 등록하기

공통 개발 질문 dev-03의 기대 문서 목록과 answerable을 채우세요.

In [6]:
# 질문, 기대 원문 ID 목록, 답변 가능 여부를 적으세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>case = {&quot;question_id&quot;: &quot;dev-03&quot;, &quot;question&quot;: &quot;주차 요금은 얼마인가요?&quot;,
        &quot;expected_document_ids&quot;: [], &quot;answerable&quot;: False}
print(case[&quot;expected_document_ids&quot;], case[&quot;answerable&quot;])</code></pre>
<p>실행 결과</p><pre><code>[] False</code></pre>
<p>빈 목록과 False는 지금 가진 자료로 주차 요금을 확인할 수 없다는 뜻입니다. 무료나 0원이라는 답을 나타내지 않습니다.</p>
</details>

## 돌아보기

월요일, 14일, 24시간 중 하나를 골라 자료 이름·쪽·원문 ID를 말해 보세요. 다음 장에 넘길 때는 세 문서의 본문과 ID를 유지하고, 실험으로 잘게 자른 자료는 별도로 식별합니다.

### 문서에서 답변까지

**문서를 읽고 정리**　본문과 원문 위치를 보존합니다.

```
library-1 · 도서 대출 기간은 14일입니다.
```

**질문 관련 문서 선택**　검색 방법은 이후 장에서 구현합니다.

```
질문: 대출 기간은?
선택할 문서: library-1
```

**근거를 입력에 포함**　근거가 없으면 답변을 보류합니다.

```
question + documents → answer
```
